## Reconstrucción tomográfica con regularización cuadrática


In [ ]:
import numpy as np
from funciones import N, xfantoma, matriz_proyeccion


## Inciso a)
def matriz_diferencias(n):
    D = np.zeros((2 * n * (n - 1), n**2))
    fila = 0

    # Diferencias horizontales: X[i, j+1] - X[i, j]
    for i in range(n):
        for j in range(n - 1):
            k = i * n + j ## posición del pixel en el vector
            D[fila, k] = -1
            D[fila, k + 1] = 1
            fila += 1

    # Diferencias verticales: X[i+1, j] - X[i, j]
    for i in range(n - 1):
        for j in range(n):
            k = i * n + j
            D[fila, k] = -1
            D[fila, k + n] = 1
            fila += 1

    return D

n = 8
D = matriz_diferencias(n)
print("Dimensiones obtenidas:", D.shape)

verificacion_filas = np.all((np.sum(D == -1, axis=1) == 1) & (np.sum(D == 1, axis=1) == 1))

print("¿Todas las filas contienen exactamente un -1 y un 1?", verificacion_filas)


Dimensiones obtenidas: (112, 64)
¿Todas las filas contienen exactamente un -1 y un 1? True


In [33]:
## Inciso b)

x_constante = np.ones(n**2) #  representa una imagen en la que todos los píxeles tienen valor 1

Dx_constante = D @ x_constante
Dx_fantoma = D @ xfantoma

print("Dx_constante:", Dx_constante)

print("¿Dx_constante es igual a cero?", np.allclose(Dx_constante, 0) )

print("Norma de Dx_constante:", np.linalg.norm(Dx_constante))
print("Norma de Dxfantoma:", np.linalg.norm(Dx_fantoma))

Dx_constante: [0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
¿Dx_constante es igual a cero? True
Norma de Dx_constante: 0.0
Norma de Dxfantoma: 2.705549851693737


Cada fila de $D$ calcula la diferencia entre dos píxeles vecinos. Como todos los píxeles de `x_constante` valen $1$, cada diferencia es $1-1=0$. Por eso, $D x_{\text{constante}}$ es el vector nulo y su norma es cero.

En cambio, el fantoma contiene regiones con distintas intensidades. En las fronteras entre estas regiones hay píxeles vecinos diferentes, de modo que algunas componentes de $D x_{\text{fantoma}}$ no son nulas. Por esta razón, $\|D x_{\text{fantoma}}\|>0$.

### Inciso c) Interpretación de la función.

La función está definida por:

$$
J_{\sigma,\lambda}(x)
=
\frac{1}{2}
\left(
\|A_7x-b_\sigma\|^2
+
\lambda\|Dx\|^2
\right).
$$

Esta función permite evaluar qué tan adecuada es una reconstrucción $x$. Está formada por dos términos: el término de ajuste a las mediciones y el término de regularización.

El primer término es

$$
\|A_7x-b_\sigma\|^2.
$$

El vector $A_7x$ contiene las mediciones que produciría la imagen reconstruida $x$, mientras que $b_\sigma$ contiene las mediciones observadas con un nivel de ruido $\sigma$. Por lo tanto, la diferencia $A_7x-b_\sigma$ es el residuo y mide qué tan bien la reconstrucción reproduce las mediciones. Cuanto menor sea su norma, mejor será el ajuste de $x$ a los datos.

El segundo término es

$$
\lambda\|Dx\|^2.
$$

El vector $Dx$ contiene las diferencias entre píxeles vecinos, primero las horizontales y luego las verticales. En consecuencia, $\|Dx\|$ mide la variación total de la imagen entre píxeles vecinos. Una norma pequeña corresponde a una imagen más suave, mientras que una norma grande indica que existen muchos cambios o cambios intensos entre píxeles cercanos.

El parámetro $\lambda$ determina la importancia de la regularización con respecto al ajuste a las mediciones:

- Si *$\lambda=0$*, no se penalizan las diferencias entre píxeles y solamente se intenta ajustar las mediciones. Esto puede hacer que la reconstrucción también reproduzca el ruido.
- Si $\lambda$ es pequeño, se penalizan moderadamente los cambios entre píxeles, buscando reducir el ruido sin perder demasiado detalle.
- Si $\lambda$ es grande, se favorecen imágenes muy suaves. Esto puede reducir las irregularidades causadas por el ruido, pero también puede borrar bordes y detalles reales.
- Si $\lambda$ es excesivamente grande, la reconstrucción puede quedar sobresuavizada o demasiado uniforme.

Por lo tanto, la minimización de $J_{\sigma,\lambda}(x)$ busca un equilibrio entre reproducir correctamente las mediciones y obtener una imagen sin variaciones espaciales excesivas.

In [ ]:
# Reconstruimos A7 usando las mismas siete direcciones
D2 = [(1, 0), (0, 1)]
D4 = D2 + [(1, 1), (1, -1)]
D7 = D4 + [(2, 1), (1, 2), (3, 1)]

A7, rayos7 = matriz_proyeccion(N, D7)

# Mediciones limpias
b7 = A7 @ xfantoma

# Mismo vector aleatorio del ejercicio sobre ruido
np.random.seed(0)
z = np.random.randn(len(b7))

# Escala de las mediciones
escala = np.sqrt(np.mean(b7**2))

# Niveles de ruido
sigmas = [0, 0.10, 0.20]

# Mediciones para cada nivel de ruido
mediciones = {}

for sigma in sigmas:
    b_sigma = b7 + sigma * escala * z
    mediciones[sigma] = b_sigma

    print("Sigma:", sigma)
    print("Norma del ruido agregado:", np.linalg.norm(b_sigma - b7))
    print()

NameError: name 'A7' is not defined